# 03 · Massachusetts demand forecasting

**Goal:** forecast monthly Massachusetts utility (LDC) gas demand, residential + commercial + industrial, 1–12 months ahead. Power-plant burn is excluded: it follows electricity markets, not heating weather (see notebook 02).

**Models**
| Model | Idea |
|---|---|
| Seasonal naive | Same month last year (benchmark) |
| HDD regression | Utility-standard weather regression on the last 10 years; coefficients are interpretable |
| SARIMAX | Regression on HDD/CDD with seasonal ARIMA errors |
| Prophet | Trend + yearly seasonality + HDD/CDD regressors, last 10 years |
| LightGBM | Gradient boosting on weather, calendar and last-year volume |
| Hybrid | HDD regression for the physics + LightGBM on its residuals |

**Evaluation design.** Expanding-window rolling-origin backtest: every 3 months from Dec 2015, train only on data up to the origin and forecast the next 12 months. Run twice:
* **Normal weather (ex-ante):** future HDD = 15-year normals known at the origin (recent winters are warmer, so 30-year normals bias forecasts high). This is the real-world forecast.
* **Actual weather (ex-post):** future HDD = observed. This isolates model skill from weather uncertainty.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, data, features, plotting
plotting.use_style()
pd.set_option("display.float_format", "{:,.2f}".format)
from src import demand_models as dm, demand_backtest as dbt
raw = data.load_raw()
dfd = features.build_demand_frame(raw["ma_consumption"], raw["weather_ma"], target=config.DEMAND_TARGET)
print(f"{len(dfd)} months, {dfd.index.min():%Y-%m} to {dfd.index.max():%Y-%m}")

## HDD regression on the full history
Read the `hdd` coefficient as MMcf of additional monthly demand per heating degree day.

In [ ]:
reg = dm.HDDRegression().fit(dfd)
print(reg.res.summary().tables[1])
print(f"Adj. R²: {reg.res.rsquared_adj:.3f}")

## Backtest

Design choices below (target, normals window, training window) were chosen by comparing backtests; the next cell reproduces the normals comparison.

In [ ]:
quick = [dm.SeasonalNaive, dm.HDDRegression, dm.SARIMAXModel, dm.LGBMModel]
rows = {}
for ny in (10, 15, 30):
    s = dbt.score(dbt.rolling_backtest(dfd, models=quick, normal_years=ny, verbose=False))
    rows[f"{ny}-yr normals"] = s["MAPE %"]
pd.DataFrame(rows).round(2)

In [ ]:
bt = dbt.rolling_backtest(dfd, weather="normal")
bt_x = dbt.rolling_backtest(dfd, weather="actual", verbose=False)
s_normal, s_actual = dbt.score(bt), dbt.score(bt_x)
s_normal

In [ ]:
compare = pd.DataFrame({"MAPE % (normal weather)": s_normal["MAPE %"],
                        "MAPE % (actual weather)": s_actual["MAPE %"]})
compare["Error due to weather (pts)"] = compare.iloc[:, 0] - compare.iloc[:, 1]
compare.sort_values("MAPE % (normal weather)")

Most of the remaining error in an ex-ante forecast is weather, not the model. That is the case for pairing the point forecast with weather scenarios.

In [ ]:
mh = dbt.mape_by_horizon(bt)
fig, ax = plt.subplots()
for m in mh.columns:
    ax.plot(mh.index, mh[m], marker="o", ms=4, color=plotting.MODEL_COLORS[m], label=m)
ax.set_xlabel("Months ahead")
ax.set_ylabel("MAPE (%)")
ax.set_title("Backtest error by forecast horizon (normal-weather forecasts)")
ax.legend(ncol=3, fontsize=8)
plt.show()

In [ ]:
best = s_normal.index[0] if s_normal.index[0] != "Seasonal naive" else s_normal.index[1]
best_cls = next(M for M in dm.ALL_MODELS if M.name == best)
one = bt_x[(bt_x["model"] == best) & (bt_x["h"] == 1)].set_index("date").sort_index()
fig, ax = plt.subplots()
ax.plot(one.index, one["y"] / 1000, color=plotting.NEUTRAL, label="Actual")
ax.plot(one.index, one["yhat"] / 1000, color=plotting.MODEL_COLORS[best], lw=1.5, label=f"{best}, 1-month ahead")
ax.set_ylabel("Bcf per month")
ax.set_title(f"Out-of-sample fit: {best}")
ax.legend()
plt.show()

## Forecast: next 12 months under three weather scenarios
Cold = 90th-percentile HDD for each month, warm = 10th percentile. The shaded band is a 90% interval from the model's own backtest errors at each horizon.

In [ ]:
fc = dbt.scenario_forecast(dfd, best_cls, bt=bt)
hist = dfd["y"].iloc[-24:]
fig, ax = plt.subplots()
ax.plot(hist.index, hist / 1000, color=plotting.NEUTRAL, label="Actual")
ax.fill_between(fc.index, fc["lo90"] / 1000, fc["hi90"] / 1000, color=plotting.SCENARIO_COLORS["normal"], alpha=.15, lw=0, label="90% interval")
for sc in ("cold", "normal", "warm"):
    ax.plot(fc.index, fc[sc] / 1000, color=plotting.SCENARIO_COLORS[sc], lw=2 if sc == "normal" else 1.5,
            ls="-" if sc == "normal" else "--", label=f"{sc.title()} weather")
ax.set_ylabel("Bcf per month")
ax.set_title(f"Massachusetts utility gas demand: 12-month forecast ({best})")
ax.legend(ncol=5, fontsize=8, loc="upper left")
plt.show()
(fc[["normal", "cold", "warm", "lo90", "hi90"]] / 1000).round(1)

In [ ]:
winter = fc[fc.index.month.isin(dbt.WINTER)]
print(f"Winter (Nov-Mar) total, normal weather: {winter['normal'].sum()/1000:,.1f} Bcf")
print(f"Cold winter adds {(winter['cold'].sum()/winter['normal'].sum()-1)*100:+.1f}%, "
      f"warm winter {(winter['warm'].sum()/winter['normal'].sum()-1)*100:+.1f}%")

## Conclusions
*Fill in after running on real data:* which model won, its MAPE vs. the seasonal naive benchmark, how much error was weather, and the winter range between warm and cold scenarios.